# Components in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

Alfred is hosting a party and needs to be able to find relevant information on personas that will be attending the party. Therefore, we will use a `QueryEngine` to index and search through a database of personas.

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
!pip install llama-index datasets llama-index-callbacks-arize-phoenix arize-phoenix llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [1]:
from huggingface_hub import login

login()

## Create a `QueryEngine` for retrieval augmented generation

### Setting up the persona database

We will be using personas from the [dvilasuero/finepersonas-v0.1-tiny dataset](https://huggingface.co/datasets/dvilasuero/finepersonas-v0.1-tiny). This dataset contains 5K personas that will be attending the party!

Let's load the dataset and store it as files in the `data` directory

In [2]:
from datasets import load_dataset, logging
from pathlib import Path

logging.set_verbosity_info()
dataset = load_dataset(path="dvilasuero/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w") as f:
        f.write(persona["persona"])

Found cached dataset finepersonas-v0.1-tiny (/Users/chongjiu/.cache/huggingface/datasets/dvilasuero___finepersonas-v0.1-tiny/default/0.0.0/877c402c4434d631b5055853bc50ba93fbdf9c12)


Awesome, now we have a local directory with all the personas that will be attending the party, we can load and index!

### Loading and embedding persona documents

We will use the `SimpleDirectoryReader` to load the persona descriptions from the `data` directory. This will return a list of `Document` objects.

In [3]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
len(documents) # 5000 train data

5000

Now we have a list of `Document` objects, we can use the `IngestionPipeline` to create nodes from the documents and prepare them for the `QueryEngine`. We will use the `SentenceSplitter` to split the documents into smaller chunks and the `HuggingFaceEmbedding` to embed the chunks.

In [4]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.node_parser import SentenceSplitter
# A processing step that converts raw docs into nodes (a unit of information that can be indexed and queried)
from llama_index.core.ingestion import IngestionPipeline

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes

2026-02-24 11:17:58,840 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-02-24 11:18:00,966 - INFO - 1 prompt is loaded, with the key: query


[TextNode(id_='5ba6a567-29ad-4f91-9699-19848416afec', embedding=[-0.016190746799111366, 0.02808922715485096, 0.0324607789516449, 0.004953076597303152, 0.007439726497977972, -0.01195851806551218, 0.02557845413684845, 0.040115080773830414, -0.059672966599464417, -0.03248359635472298, 0.011018901132047176, -0.03690111264586449, -0.031023776158690453, 0.02739700861275196, -0.00897024292498827, -0.024445921182632446, 0.006906474009156227, 0.07338818907737732, 0.022401416674256325, 0.00428661098703742, 0.003625687910243869, -0.0710131824016571, 0.04377647116780281, -0.0353805348277092, -0.0325239859521389, -0.017985874786973, 0.0017925004940479994, -0.032628800719976425, -0.013057371601462364, -0.12729260325431824, -0.052950017154216766, 0.02325405180454254, -0.028538133949041367, 0.023580402135849, 0.07222572714090347, 0.024904457852244377, 0.008675217628479004, 0.03490820527076721, 0.012613100931048393, 0.010338105261325836, 0.030998563393950462, 0.035164933651685715, -0.000636016309726983

As, you can see, we have created a list of `Node` objects, which are just chunks of text from the original documents. Let's explore how we can add these nodes to a vector store.

### Storing and indexing documents

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it.
In this case, we will use `Chroma` to store our documents.
Let's run the pipeline again with the vector store attached.
The `IngestionPipeline` caches the operations so this should be fast!

In [5]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

# vector database on disk
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

2026-02-24 11:18:06,104 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-24 11:18:06,182 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-02-24 11:18:07,954 - INFO - 1 prompt is loaded, with the key: query


10

We can create a `VectorStoreIndex` from the vector store and use it to query the documents by passing the vector store and embedding model to the `from_vector_store()` method.

In [6]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# Use BAAI/bge-small-en-v1.5 as the "translater" that converts natural language input -> vector embeddings
# Note, the "model tax" here is relatively small. converting words -> vector is pretty fast
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

2026-02-24 11:18:15,282 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-02-24 11:18:17,147 - INFO - 1 prompt is loaded, with the key: query


We don't need to worry about persisting the index to disk, as it is automatically saved within the `ChromaVectorStore` object and the passed directory path.

### Querying the index

Now that we have our index, we can use it to query the documents.
Let's create a `QueryEngine` from the index and use it to query the documents using a specific response mode.


In [8]:
# This block is the heart of RAG!
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import nest_asyncio

# Hack: python asyncio, the lib that runs judyter, doesn't allow one loop to run inside another
# This hack patches the environment so LlamaIndex can run its async tasks
nest_asyncio.apply()  

# Use qwen as the brain
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# hook qwen with our index (vector db)
query_engine = index.as_query_engine(
    llm=llm,

    # don't just dump data back to llm, summarize it nicely
    response_mode="tree_summarize",
)

response = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)
response

Response(response="An individual deeply engaged in the study of Cypriot culture, history, and society, having dedicated significant time to research and reside in Cyprus. This person's expertise allows for a profound understanding of the local customs and way of life, making them a valuable resource for anyone seeking insights into Cypriot traditions and social dynamics.", source_nodes=[NodeWithScore(node=TextNode(id_='2a96b32a-403b-42f7-a9cd-a5a6f018c428', embedding=None, metadata={'file_path': '/Users/chongjiu/Desktop/agent-course/notebooks/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-02-24', 'last_modified_date': '2026-02-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelatio

## Evaluation and observability

LlamaIndex provides **built-in evaluation tools to assess response quality.**
These evaluators leverage LLMs to analyze responses across different dimensions.
We can now check if the query is faithful to the original persona.

In [9]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

False

If one of these LLM based evaluators does not give enough context, we can check the response using the Arize Phoenix tool, after creating an account at [LlamaTrace](https://llamatrace.com/login) and generating an API key.

In [ ]:
import llama_index
import os

PHOENIX_API_KEY = "<PHOENIX_API_KEY>"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
llama_index.core.set_global_handler(
    "arize_phoenix", endpoint="https://llamatrace.com/v1/traces"
)


Now, we can query the index and see the response in the Arize Phoenix tool.

In [10]:
response = query_engine.query(
    "What is the name of the someone that is interested in AI and techhnology?"
)
response

Response(response='The information provided does not mention anyone interested in AI and technology. It describes two individuals: one with expertise in Cypriot culture, history, and society, and another who is a pulmonologist with an interest in educating patients about respiratory issues. There is no reference to AI or technology in the given details.', source_nodes=[NodeWithScore(node=TextNode(id_='2a96b32a-403b-42f7-a9cd-a5a6f018c428', embedding=None, metadata={'file_path': '/Users/chongjiu/Desktop/agent-course/notebooks/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-02-24', 'last_modified_date': '2026-02-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>

We can then go to the [LlamaTrace](https://llamatrace.com/login) and explore the process and response.

![arize-phoenix](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/arize.png)    